In [1]:
from pathlib import Path

from copy import deepcopy
from time import perf_counter
import importlib

import field_config
import synthetic_fields

# Pick up changes made to either Python file.
importlib.reload(field_config)
importlib.reload(synthetic_fields)

config = field_config.config

from synthetic_fields import generate_cube, save_cube, get_grid_tag

check config file

In [2]:
print(config)

{'field_synth': 'velocity', 'model_gs': 'stable', 'alpha': 1.5, 'seed': 19880127, 'shape': (64, 64, 64), 'spacing': (1.0, 1.0, 1.0), 'len_scale': (16.0, 16.0, 16.0), 'angles': (0.0, 0.0, 0.0)}


In [3]:
# Models to include in this batch.
models = ("exponential", "gaussian", "stable")

# Shape parameters to use only for Stable.
stable_alphas = (1.0, 1.5, 2.0)

# One seed per field role, reused across the models.
field_seeds = {
    "velocity": 19880127,
    "density": 19970221,
}

# Output settings.
output_dir = Path("cubes") / get_grid_tag(config)
overwrite = True

print(f"Output folder: {output_dir}")

Output folder: cubes/N64_L16


In [4]:
configurations = []

for model_name in models:

    # Exponential and Gaussian each need one configuration per field.
    # Stable needs one configuration per alpha and field.
    alpha_values = stable_alphas if model_name == "stable" else (None,)

    for alpha in alpha_values:
        for field_role, seed in field_seeds.items():
            cube_config = deepcopy(config)

            cube_config.update(
                model_gs=model_name,
                field_synth=field_role,
                seed=seed,
            )

            if model_name == "stable":
                cube_config["alpha"] = alpha
            else:
                cube_config.pop("alpha", None)

            configurations.append(cube_config)

In [5]:
print(f"Configurations prepared: {len(configurations)}\n")

for index, cube_config in enumerate(configurations, start=1):
    print(
        f"{index:2d}. "
        f"{cube_config['model_gs']:12s} "
        f"{cube_config['field_synth']:8s} "
        f"alpha={str(cube_config.get('alpha', 'fixed')):5s} "
        f"seed={cube_config['seed']} "
        f"shape={cube_config['shape']}"
    )

Configurations prepared: 10

 1. exponential  velocity alpha=fixed seed=19880127 shape=(64, 64, 64)
 2. exponential  density  alpha=fixed seed=19970221 shape=(64, 64, 64)
 3. gaussian     velocity alpha=fixed seed=19880127 shape=(64, 64, 64)
 4. gaussian     density  alpha=fixed seed=19970221 shape=(64, 64, 64)
 5. stable       velocity alpha=1.0   seed=19880127 shape=(64, 64, 64)
 6. stable       density  alpha=1.0   seed=19970221 shape=(64, 64, 64)
 7. stable       velocity alpha=1.5   seed=19880127 shape=(64, 64, 64)
 8. stable       density  alpha=1.5   seed=19970221 shape=(64, 64, 64)
 9. stable       velocity alpha=2.0   seed=19880127 shape=(64, 64, 64)
10. stable       density  alpha=2.0   seed=19970221 shape=(64, 64, 64)


In [6]:
# Reset the output list each time this cell runs.
saved_paths = []
batch_start = perf_counter()

for index, cube_config in enumerate(configurations, start=1):
    start = perf_counter()

    label = (
        f"{cube_config['model_gs']} | "
        f"{cube_config['field_synth']}"
    )

    if cube_config["model_gs"] == "stable":
        label += f" | alpha={cube_config['alpha']:g}"

    print(
        f"[{index}/{len(configurations)}] Generating {label}",
        flush=True,
    )

    field_3d, grid, model, m3d = generate_cube(cube_config)

    fits_path = save_cube(
        field_3d,
        cube_config,
        m3d,
        output_dir=output_dir,
        overwrite=overwrite,
    )

    saved_paths.append(fits_path)

    print(
        f"  Saved: {fits_path.name}\n"
        f"  Time: {perf_counter() - start:.1f} s",
        flush=True,
    )

    # Release this sample before generating the next one.
    del field_3d, grid, model

print(
    f"\nCompleted: {len(saved_paths)} cubes saved.\n"
    f"Total time: {perf_counter() - batch_start:.1f} s"
)

[1/10] Generating exponential | velocity
  Saved: exponential_3d_velocity_N64_L16.fits
  Time: 3.3 s
[2/10] Generating exponential | density
  Saved: exponential_3d_density_N64_L16.fits
  Time: 3.3 s
[3/10] Generating gaussian | velocity
  Saved: gaussian_3d_velocity_N64_L16.fits
  Time: 3.2 s
[4/10] Generating gaussian | density
  Saved: gaussian_3d_density_N64_L16.fits
  Time: 3.2 s
[5/10] Generating stable | velocity | alpha=1
  Saved: stable_alpha1_3d_velocity_N64_L16.fits
  Time: 3.7 s
[6/10] Generating stable | density | alpha=1
  Saved: stable_alpha1_3d_density_N64_L16.fits
  Time: 3.7 s
[7/10] Generating stable | velocity | alpha=1.5
  Saved: stable_alpha1.5_3d_velocity_N64_L16.fits
  Time: 3.3 s
[8/10] Generating stable | density | alpha=1.5
  Saved: stable_alpha1.5_3d_density_N64_L16.fits
  Time: 3.5 s
[9/10] Generating stable | velocity | alpha=2
  Saved: stable_alpha2_3d_velocity_N64_L16.fits
  Time: 3.2 s
[10/10] Generating stable | density | alpha=2
  Saved: stable_alpha2

In [7]:
from astropy.io import fits

print(
    f"{'Model':12s} {'Field':8s} {'M3D':>5s} "
    f"{'Min':>10s} {'Max':>10s} {'Mean':>10s} "
    f"{'Std':>10s} {'Variance':>10s}"
)

for path in saved_paths:
    header = fits.getheader(path)

    print(
        f"{header['MODEL']:12s} "
        f"{header['FIELD']:8s} "
        f"{header['M3D']:5.2f} "
        f"{header['DATAMIN']:10.4f} "
        f"{header['DATAMAX']:10.4f} "
        f"{header['MEAN']:10.4f} "
        f"{header['STD']:10.4f} "
        f"{header['STD'] ** 2:10.4f}"
    )

Model        Field      M3D        Min        Max       Mean        Std   Variance
exponential  velocity  1.00    -3.4632     3.1256    -0.3626     0.7976     0.6362
exponential  density   1.00    -2.9442     3.4235     0.1190     0.8087     0.6539
gaussian     velocity  2.00    -2.6302     2.4502    -0.1659     0.9452     0.8934
gaussian     density   2.00    -3.2418     2.3904    -0.2377     0.8619     0.7428
stable       velocity  1.00    -4.4930     3.9113    -0.1649     0.9214     0.8489
stable       density   1.00    -4.2528     4.6636     0.0722     0.9755     0.9516
stable       velocity  1.50    -3.7811     3.5600     0.1442     1.0050     1.0100
stable       density   1.50    -3.6706     3.7578    -0.0231     0.8707     0.7581
stable       velocity  2.00    -2.6282     2.4492    -0.0275     0.9107     0.8294
stable       density   2.00    -3.2491     2.3877    -0.2486     0.8958     0.8025
